# 03 Trees and ensembles

Role 3. Due Friday 20 Nov (first tree on Thursday).

One tree and why it overfits, two ways to prune it, then bagging and random forests (less variance) and boosting (less bias), all compared with matrix pricing on valid.

Class notebooks to follow: `RTrees_boston` (steps 2 to 5), `Regression_Improve Bagg & RF with Boosting` (steps 6, 8 to 10), `RF_risk` (step 7).

This is a starter: the setup cell runs as it is, and each step of your walkthrough has a heading and an empty
cell below. Fill the cells in order, following the class notebook.

Rules: validation only (`partition == "valid"`); test is opened once, in `99_final`. Shared code comes from
`ml2` and is never copied here. Figures go to `reports/figures/03_<what>.png` with units on the axes
(bp, years). Every run, including failures and dropped ideas, gets one line in `RUN_LOG.md`.

## 1. Setup

Trees need no scaling: `make_preprocessor(scale=False, one_hot=False)`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline

from ml2.baseline import matrix_same_date
from ml2.config import load_config, resolve
from ml2.data import check_panel, load_panel
from ml2.evaluate import metrics_by_stratum, regression_metrics
from ml2.features import make_preprocessor, xy
from ml2.splits import issuer_group_folds, walk_forward_folds

cfg = load_config()
TARGET = cfg["problem"]["target"]
FIG = resolve("reports/figures")
FIG.mkdir(parents=True, exist_ok=True)

check_panel()
df = load_panel(partitions=["train", "valid"])
train, valid = df[df.partition == "train"], df[df.partition == "valid"]
X_tr, y_tr = xy(train)
X_va, y_va = xy(valid)

# Searches run on a 30% sample of train (time order kept); chosen models are refitted on all of train
sub = train.sample(frac=0.3, random_state=42).sort_values("snapshot_date")
X_s, y_s = xy(sub)
cv = walk_forward_folds(sub.loc[X_s.index, "snapshot_date"])  # replaces cv=5 / KFold in the class notebooks
print(f"train {X_tr.shape}, valid {X_va.shape}, search sample {X_s.shape}, {len(cv)} time-ordered folds")


def tree_pipe(est):
    return Pipeline([("pre", make_preprocessor(scale=False, one_hot=False)), ("est", est)])

In [ ]:
# The bar: matrix pricing re-marked on each date (the gate baseline), on valid. About 1.5 minutes.
valid = valid.copy()
valid["pred_matrix_same_date"] = matrix_same_date(valid)


def vs_bar(pred, name="model"):
    """MAE by grade for predictions on X_va's rows, next to the gate baseline on the same rows."""
    rows = valid.loc[X_va.index].assign(pred=np.asarray(pred))
    out = pd.concat({
        name: metrics_by_stratum(rows, TARGET, "pred", "grade_bucket").set_index("stratum")["mae_bps"],
        "matrix re-marked (bar)": metrics_by_stratum(
            rows, TARGET, "pred_matrix_same_date", "grade_bucket").set_index("stratum")["mae_bps"],
    }, axis=1).T
    return out[["IG", "HY", "ALL"]].round(1)


vs_bar(valid.loc[X_va.index, "pred_matrix_same_date"], "matrix re-marked")  # check: both rows equal

## 2. Initial tree

`RTrees_boston`, single tree. Train MAE against CV MAE: show the overfitting.

In [ ]:
# Step 2: Initial tree

## 3. Reducing the complexity

`GridSearchCV` over `max_depth` and `min_samples_leaf`, `scoring="neg_mean_absolute_error"`, `cv=cv`; CV MAE heatmap.

In [ ]:
# Step 3: Reducing the complexity

## 4. Cost-complexity pruning, 1-SE rule

About 25 alphas spread over the pruning path; largest alpha within one standard error of the best.

In [ ]:
# Step 4: Cost-complexity pruning, 1-SE rule

## 5. Comparison between the 3 trees

Same folds for all; CV MAE and number of leaves; `plot_tree` of the chosen tree to depth 2 or 3.

In [ ]:
# Step 5: Comparison between the 3 trees

## 6. Bagging

`RandomForestRegressor(max_features=None, oob_score=True)`. Why OOB is optimistic here.

In [ ]:
# Step 6: Bagging

## 7. How many trees do we need?

`RF_risk`: OOB MAE as trees are added with `warm_start=True`.

In [ ]:
# Step 7: How many trees do we need?

## 8. Random forest

Same grid plus `max_features` (`"sqrt"`, 0.3, 0.5).

In [ ]:
# Step 8: Random forest

## 9. Boosting: XGBoost, then LightGBM

`n_estimators=500`, `subsample=0.8`, `colsample_bytree=0.7`; grids on `learning_rate` and `max_depth` (XGBoost), `num_leaves` and `max_depth` (LightGBM).

In [ ]:
# Step 9: Boosting: XGBoost, then LightGBM

## 10. Interpretation

Impurity importances, permutation importance on valid, SHAP for the best boosting model.

In [ ]:
# Step 10: Interpretation

## 11. Comparison table

Every model with `vs_bar(...)`: valid MAE overall, IG and HY against the gate baseline.

In [ ]:
# Step 11: Comparison table

## 12. New-issuer question

The best model with `issuer_group_folds(sub.loc[X_s.index, "issuer_ticker"])`: error for companies never seen in training.

In [ ]:
# Step 12: New-issuer question

## 13. Hand-off

Chosen model and parameters to RUN_LOG and to Miguel for `99_final`. Do not score test.

In [ ]:
# Step 13: Hand-off

## RUN_LOG

One line per run in `notebooks/RUN_LOG.md`:

`| date | who | notebook | model and key parameters | features | fit on train, scored on valid | MAE bps (IG / HY) | notes |`